# Real-World Evidence on OMOP CDM
**Cohort analytics on MIMIC-IV demo data (work in progress)**

This notebook explores the MIMIC-IV demo dataset (100 ICU patients) converted to the OMOP Common Data Model, using SQL (DuckDB) and Python.

**Data source:** MIMIC-IV demo data in the OMOP CDM, v0.9 (PhysioNet, Open Database License). Data are not included in this repository: download them from PhysioNet and place the `1_omop_data_csv/` folder inside `data/`.

## 1. Setup

In [6]:
# Libraries
import duckdb
import glob
import os

# Path to the OMOP CSV files
DATA = "../data/1_omop_data_csv"

## 2. Load OMOP tables as DuckDB views
Each CSV file becomes a named view, so tables can be queried directly with SQL (e.g. `FROM person`).

In [7]:
# Create one view per CSV file
for path in glob.glob(f"{DATA}/*.csv"):
    name = os.path.basename(path).replace(".csv", "")
    duckdb.sql(f'''CREATE OR REPLACE VIEW "{name}" AS SELECT * FROM read_csv_auto('{path}')''')

# List all tables
duckdb.sql("SHOW TABLES")

nameVARCHAR
2b_concept
2b_concept_relationship
2b_vocabulary
attribute_definition
care_site
cdm_source
cohort
cohort_attribute
cohort_definition
condition_era


## 3. Number of patients

In [8]:
# Count patients in the person table
duckdb.sql("SELECT COUNT(*) AS n_patients FROM person")

n_patientsBIGINT
100


## 4. Explore the person table
Note: MIMIC-IV shifts dates into the future (e.g. birth years after 2100) to protect patient identity. A `concept_id` of 0 means the source value could not be mapped to a standard concept.

In [9]:
# First 5 rows
duckdb.sql("SELECT * FROM person LIMIT 5")

person_idBIGINT,gender_concept_idBIGINT,year_of_birthBIGINT,month_of_birthVARCHAR,day_of_birthVARCHAR,birth_datetimeVARCHAR,race_concept_idBIGINT,ethnicity_concept_idBIGINT,location_idVARCHAR,provider_idVARCHAR,care_site_idVARCHAR,person_source_valueBIGINT,gender_source_valueVARCHAR,gender_source_concept_idBIGINT,race_source_valueVARCHAR,race_source_concept_idBIGINT,ethnicity_source_valueVARCHAR,ethnicity_source_concept_idBIGINT
3589912774911670296,8507,2095,NULL,NULL,NULL,0,38003563,NULL,NULL,NULL,10009628,M,0,NULL,0,HISPANIC/LATINO,2000001408
-3210373572193940939,8507,2079,NULL,NULL,NULL,0,38003563,NULL,NULL,NULL,10011398,M,0,NULL,0,HISPANIC/LATINO,2000001408
-775517641933593374,8507,2149,NULL,NULL,NULL,8516,0,NULL,NULL,NULL,10004235,M,0,BLACK/AFRICAN AMERICAN,2000001406,NULL,0
-2575767131279873665,8507,2050,NULL,NULL,NULL,8516,0,NULL,NULL,NULL,10024043,M,0,BLACK/AFRICAN AMERICAN,2000001406,NULL,0
-8970844422700220177,8507,2114,NULL,NULL,NULL,8527,0,NULL,NULL,NULL,10038933,M,0,WHITE,2000001404,NULL,0


## 5. Patients by sex: standard concept vs. source value

In [10]:
# Patients by sex
duckdb.sql("""
SELECT gender_concept_id, gender_source_value, COUNT(*) AS n
FROM person
GROUP BY gender_concept_id, gender_source_value
""")

gender_concept_idBIGINT,gender_source_valueVARCHAR,nBIGINT
8507,M,57
8532,F,43


## 6. Size of the measurement table

In [12]:
# Total measurements and average per patient
duckdb.sql("""
SELECT COUNT(*) AS n_measurements,
       COUNT(*) / 100 AS avg_per_patient
FROM measurement
""")

n_measurementsBIGINT,avg_per_patientDOUBLE
338550,3385.5


## 7. Visits per patient (LEFT JOIN)

In [13]:
# Number of visits per patient, keeping all patients
duckdb.sql("""
SELECT p.person_id,
       p.gender_source_value AS sex,
       COUNT(v.visit_occurrence_id) AS n_visits
FROM person AS p
LEFT JOIN visit_occurrence AS v
       ON p.person_id = v.person_id
GROUP BY p.person_id, p.gender_source_value
ORDER BY n_visits DESC
LIMIT 10
""")

person_idBIGINT,sexVARCHAR,n_visitsBIGINT
-6225647829918357531,F,102
4783904755296699562,M,92
7155255168997124770,M,71
1741351032930224901,F,42
7918537411740862407,F,38
4985579811051920670,F,31
7131048714591189903,M,28
3129727379702505063,F,28
-626229666378242477,M,26
1258460361496302149,F,26


## 8. Visit types
Total visits per standard `visit_concept_id`. Concept names will be added once the full OMOP vocabulary (Athena) is loaded.

In [15]:
# How many visits of each type?
duckdb.sql("""
SELECT visit_concept_id,
       COUNT(*) AS n
FROM visit_occurrence
GROUP BY visit_concept_id
ORDER BY n DESC
""")

visit_concept_idBIGINT,nBIGINT
38004207,582
8870,119
581385,82
262,38
8883,18
9201,13
